# MLP with Tensor Flow

---

## Multi-Layer Perceptron Model

In this task, we implement and evaluate a Multi-Layer Perceptron (MLP) model using TensorFlow. We will use a simple MLP architecture with one hidden layer and test how varying the hidden layer size affects model performance on the MNIST dataset. The MNIST dataset consists of grayscale images of handwritten digits (0-9), and our goal is to classify each image correctly.

Using TensorFlow for this task allows us to define, train, and evaluate MLP models efficiently, providing insights into how model capacity (controlled by hidden layer size) impacts accuracy and generalization.

**Explanation:**

- **1. Data Preparation**:
    - Use the MNIST dataset, with one-hot encoded labels, to train and evaluate the model.

     We will further work with the MNIST dataset (Modified National Institute of Standards and Technology) which is a classic dataset widely used for training and testing machine learning models in image classification tasks. It consists of 70,000 grayscale images of handwritten digits from 0 to 9, split into 60,000 training images and 10,000 test images.

Each image in the dataset is 28x28 pixels, unrolled into a 784-dimensional vector for modeling (our input space).

- **2. Model Definition and Compilation**:
    - Define an MLP model in TensorFlow with:
      - A single hidden layer of specified size.
      - `relu` activation for the hidden layer.
      - `softmax` activation for the output layer (for 10-class classification).
    - Compile the model with the Adam optimizer and categorical cross-entropy loss.
  
- **3. Train the Model**:
    - Train the model on the MNIST training data for a set number of epochs (e.g., 10).
    - Monitor both training and validation loss and accuracy throughout training.

- **4. Evaluate Model Performance**:
    - After training, calculate and display a confusion matrix for the test set to understand the classification performance across different classes.
    - Visualize the loss and accuracy trends during training to assess convergence and potential overfitting.

By analyzing the model's performance across different configurations, this task demonstrates the impact of hidden layer size and provides a foundational understanding of how MLPs learn to classify images in TensorFlow.


# 1. Data Preperation

In [ ]:
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Load MNIST dataset
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
# The astype forces a conversion to a class as an int (not a string)
X, y = mnist["data"], mnist["target"].astype(int)


def load_all_classes(X, y, num_train=60000, num_test=10000):
    """
    Load MNIST data for all digits (0-9) and split into train and test sets with one-hot encoding.
    
    Parameters:
    - X: np.array, feature data for all samples (e.g., from MNIST)
    - y: np.array, target labels for all samples (e.g., from MNIST)
    - num_train: int, number of samples to return in the training set (default: 60000)
    - num_test: int, number of samples to return in the test set (default: 10000)
    
    Returns:
    - X_train, y_train: training data and one-hot encoded labels
    - X_test, y_test: test data and one-hot encoded labels
    """
    # One-hot encode the labels
    encoder = OneHotEncoder(sparse_output=False, categories='auto')
    y_one_hot = encoder.fit_transform(y.reshape(-1, 1))

    # Shuffle and split into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y_one_hot, train_size=num_train, test_size=num_test, random_state=42, stratify=y)

    return X_train, y_train, X_test, y_test

# Use the function to load and split the MNIST dataset
X_train, y_train, X_test, y_test = load_all_classes(X, y, num_train=1000, num_test=1000)

# Display shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

In [ ]:
#Alternative


from tensorflow import keras

# Load MNIST data set
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Preprocess the data (these are NumPy arrays)
x_train = x_train.reshape(60000, 784).astype('float32') / 255
x_test = x_test.reshape(10000, 784).astype('float32') / 255

# Reserve 10,000 samples for validation
x_val = x_train[-10000:]
y_val = y_train[-10000:]
x_train = x_train[:-10000]
y_train = y_train[:-10000]

# 2. Model Architechture

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Define the model architecture
model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(X_train.shape[1],)),  # Input layer
    tf.keras.layers.Dense(32, activation='relu'),                 # Hidden layer with 64 units
    tf.keras.layers.Dense(10, activation='softmax')               # Output layer with 10 units for 10 classes
])

# Compile the model with optimizer, loss function, and metrics
model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Train the model and store training history
history = model.fit(X_train, y_train, 
                    epochs=20,  # You can adjust the number of epochs as needed
                    batch_size=32, 
                    validation_data=(X_test, y_test), 
                    verbose=1)

# Plot the loss and accuracy over epochs
plt.figure(figsize=(12, 5))

# Plot training and validation loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label="Training Loss")
plt.plot(history.history['val_loss'], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss During Training")
plt.legend()

# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label="Training Accuracy")
plt.plot(history.history['val_accuracy'], label="Validation Accuracy", color="orange")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy During Training")
plt.legend()

plt.tight_layout()
plt.show()


### Experiment: Evaluating the Impact of Hidden Layer Size on MLP Performance

In this experiment, we aim to understand how the size of the hidden layer affects the performance of a simple Multi-Layer Perceptron (MLP) on the MNIST digit classification task. We will train MLP models with varying numbers of neurons in the hidden layer and measure both training and testing accuracy after 10 epochs for each configuration.

By comparing different hidden layer sizes, we hope to gain insight into the model's ability to capture patterns in the data with limited complexity and evaluate its generalization performance.

**Tasks:**
- **1. Define Hidden Layer Sizes**: Specify a list of hidden layer sizes to test: `[1, 2, 4, 8, 16, 32, 64, 128]`.
  
- **2. Model Setup and Compilation**:
    - For each hidden layer size, define an MLP model with:
      - One hidden layer of the specified size.
      - `relu` activation for the hidden layer.
      - `softmax` activation for the output layer (for 10-class classification).
    - Compile the model with the Adam optimizer and categorical cross-entropy loss.
  
- **3. Train the Model**: Train each model for 10 epochs on the MNIST training set, using a batch size of 32.

- **4. Evaluate Accuracy**:
    - Calculate both the training and test accuracy for each model configuration.
    - Store the accuracies to compare how hidden layer size impacts model performance.
  
- **5. Visualize Results**:
    - Plot the training and test accuracy for each hidden layer size to visualize the relationship between hidden layer size and model performance.

This approach will allow us to analyze the trade-offs in model complexity and generalization across various hidden layer sizes.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Define hidden layer sizes to test, starting with 1 and 2
hidden_layer_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
train_accuracies = []
test_accuracies = []

# TODO: Based on the code from above, 
# - vary the hidden layer size 
# - for each one build a model, train it (try only a couple epochs)
# - append the accuracies to the different lists 
# Loop over different hidden layer sizes



# Plot the results
plt.figure(figsize=(10, 5))
plt.plot(hidden_layer_sizes, train_accuracies, marker='o', label="Train Accuracy")
plt.plot(hidden_layer_sizes, test_accuracies, marker='o', label="Test Accuracy")
plt.xlabel("Hidden Layer Size")
plt.ylabel("Accuracy")
plt.title("Train and Test Accuracy for Different Hidden Layer Sizes (10 Epochs)")
plt.xticks(hidden_layer_sizes)
plt.legend()
plt.grid()
plt.show()


### Analyzing Model Complexity by Hidden Layer Size

In this task, we aim to understand how the number of parameters in a Multi-Layer Perceptron (MLP) model changes with different hidden layer sizes. By testing a variety of hidden layer sizes on an MLP for the MNIST dataset, we can evaluate the trade-offs in model complexity, computation requirements, and potential learning capacity.

**Explanation of Parameter Calculation**

Each MLP model in this experiment has:
1. **Input Layer to Hidden Layer**: The number of parameters (weights and biases) connecting the input layer to the hidden layer can be calculated as:
   $$
   \text{Parameters} = (\text{Input Dimension} + 1) \times \text{Hidden Layer Size}
   $$
   where `+1` accounts for the bias term.

2. **Hidden Layer to Output Layer**: The number of parameters connecting the hidden layer to the output layer (with 10 output neurons for MNIST’s 10 classes) is calculated as:
   $$
   \text{Parameters} = (\text{Hidden Layer Size} + 1) \times \text{Output Dimension}
   $$
   where `+1` accounts for the bias term in the output layer.

3. **Total Parameters**: The total number of parameters is the sum of the parameters from the input-to-hidden and hidden-to-output connections.

**Tasks:**

1. **Loop through Hidden Layer Sizes**: Define a list of hidden layer sizes to test, such as `[1, 2, 4, 8, 16, 32, 64, 128]`.

2. **Define and Compile Model**:
    - For each hidden layer size, define an MLP model with:
      - One hidden layer of the specified size.
      - `relu` activation for the hidden layer.
      - `softmax` activation for the output layer (for 10-class classification).
    - Use TensorFlow’s `.summary()` method to display the number of parameters in each layer and confirm calculations.

3. **Calculate and Store Parameters**:
    - Use `model.count_params()` to calculate the total number of parameters for each model configuration.

4. **Plot the Results**:
    - Visualize the total number of parameters for each hidden layer size to understand how model complexity scales with layer size.

This approach provides a clear visualization of how hidden layer size influences model complexity, helping us analyze the trade-offs between model capacity and computational cost.


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

# Define the input and output dimensions for MNIST
input_dim = 784  # 28x28 images flattened
output_dim = 10  # 10 classes for digits 0-9

# Define hidden layer sizes to test
hidden_layer_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
total_params_list = []

# Loop through each hidden layer size, create a model, and calculate parameters
for hidden_size in hidden_layer_sizes:
    # TODO
    # Calculate for each layer size how many neurons are required 

    # As an alternative: Simply rebuild the model and inspect these
    # Compare the values
    # Define the model with one hidden layer of `hidden_size`
    model = tf.keras.models.Sequential([
        tf.keras.layers.InputLayer(input_shape=(input_dim,)),
        tf.keras.layers.Dense(hidden_size, activation='relu'),
        tf.keras.layers.Dense(output_dim, activation='softmax')
    ])
    
    # Display the model summary to see the parameters for each layer
    print(f"\nModel with hidden layer size {hidden_size}:")
    model.summary()
    
    # Calculate total parameters using the model's .count_params() method
    total_params = model.count_params()
    total_params_list.append(total_params)

# Plot the total number of parameters for each hidden layer size
plt.figure(figsize=(10, 5))
plt.plot(hidden_layer_sizes, total_params_list, marker='o')
plt.xlabel("Hidden Layer Size")
plt.ylabel("Total Parameters")
plt.title("Total Parameters for Different Hidden Layer Sizes")
plt.xticks(hidden_layer_sizes)
plt.grid()
plt.show()


# Programming a neural network layer & Scores

[Keras](https://keras.io) is a high-level deep-learning framework building on top of [TensorFlow](https://www.tensorflow.org). These frameworks follow the _symbol-to-symbol derivatives_ approach, i.e. automatically derive a computational graph to calculate derivatives. You just need to declare your inputs as TensorFlow variables and use TensorFlow operations on them to compute the forward pass.  

## Task 1

Work through the [Keras tutorial on custom layers](https://keras.io/guides/making_new_layers_and_models_via_subclassing) to learn how to create your own neural network layer.  
Create a custom Keras layer that computes Gaussian basis functions, i.e. a layer that maps an input vector $\mathbf x \in \mathbb R^n$ onto an output vector $\mathbf y = f(\mathbf x) \in \mathbb R^m$ as follows:
\begin{align}
  f: \mathbf x \in \mathbb R^n \mapsto \left[w_i \exp\left(-\frac{\|\mathbf x - \boldsymbol\mu_i\|^2}{\sigma_i^2}\right)\right]_{i=1..m} \in \mathbb R^m
\end{align}

Instead of projecting an input $\mathbf x$ onto a weight vector $\mathbf w$ as the standard neuron function $f(\mathbf x) = \sigma(\mathbf w \cdot \mathbf x + b)$ does, the Gaussian basis function becomes active (with weight $w_i$) for all inputs $\mathbf x$ close to a prototype $\boldsymbol \mu_i$. This activation quickly decays with increasing distance of $\mathbf x$ to $\boldsymbol \mu_i$. The parameter $\sigma_i$ controls the width of the Gaussian, i.e. the size of the active region.

For efficient tensor-based operations you need to correctly _broadcast_ the tensors for the difference operation: TensorFlow will pass an input matrix of shape `(batch size, input dim)` for $\mathbf X$, while you will have a matrix of centers $\boldsymbol \mu$ of shape `(input dim, #units)`. To correctly [broadcast](https://numpy.org/doc/stable/user/basics.broadcasting.html) them together, you will need Keras' [`expand_dims()`](https://www.tensorflow.org/api_docs/python/tf/keras/backend/expand_dims) function to extend $\mathbf X$'s shape to `(batch size, input dim, 1)`:

In [ ]:
import numpy
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K

class RBFLayer(keras.layers.Layer):
    # TODO: Add initializer parameters to init
    def __init__(self, **kwargs):
        super(RBFLayer, self).__init__(**kwargs)
        # TODO: Do something with the additional parameters

    def build(self, input_shape):
        super(RBFLayer, self).build(input_shape)
        # TODO: Initialize properties depending on `input_shape`

    def call(self, X):
        # TODO: Process input and return
        return X


class InitCentersRandom(keras.initializers.Initializer):
    """ Initializer to initialize centers of RBF network from random samples of the data set."""

    def __init__(self, X):
        self.X = X

    def __call__(self, shape, dtype=None):
        # TODO: Generate and return initialization
        pass


### dummy example to compute diffs between data X and centers mu
X = tf.ones((3, 5))  # input tensor X with batch dimension 3 and data dim N=5
mu = tf.ones((5, 2))  # tensor mu with data dim N=5 and 2 units
diffs = K.expand_dims(X) - mu  # diffs tensor: 3 x 5 x 2
print(X.shape, mu.shape, diffs.shape)

## Task 2

Compare the performance of such a Gaussian basis function layer with that of a standard [`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense) layer on the MNIST dataset.  
Hint: Utilize existing tutorials on setting up your first MNIST MLP with Keras, e.g. https://keras.io/guides/training_with_built_in_methods.

To achieve decent performance, you want to:
- Initialize the centers $\boldsymbol \mu_i$ from random data samples $\mathbf x$ (create a custom [initializer](https://www.tensorflow.org/api_docs/python/tf/keras/initializers/Initializer))
- Initialize $\sigma_i$ to the typical in-class distance between data points.  
  Use [`scipy.spatial.distance_matrix`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance_matrix.html) to compute this statistics on a random selection of your input data.  
  (Doing it on the full dataset will probably exhaust your memory.)
- Initialize $w_i = 1$

Questions:
- How many parameters each of those networks have?
- Which network trains faster / easier?